In [ ]:
#all needed functions and importations from liberary's
import numpy as np
import matplotlib.pyplot as plt
import h5py
import heapq
from scipy import ndimage
import os

#This function plot the binairy matrix
def segmentatie_plot(data):
        data_new= np.where(data==1,(255,0,138),data)

        img= data_new/255

        plt.imshow(img)
        plt.axis("off")
        #If you want to save your segmentationplot you need to remove the # in the row below and give it a name
        #plt.savefig(f'{output_path}\{well}_{day}_{name}.png', bbox_inches= 'tight', pad_inches=0)
        plt.show()

#This function calculates the fraction of found cells
def fraction(data):
        data_new = np.where(data==2,0,data)
        rows, columns, depth= data.shape
        total= rows*columns
        fractie= data_new.sum()/total #Because the only values are zero or 1 the sum of everything can be divided by the total area
        return rows, columns, total,fractie

#This generates a binary matrix where cells are evely distrubeted in the well.
#This function does not consider the electrodes
def random_matrix(total, fraction, length, width):
    amount= round(total*fraction)
    matrix= np.zeros((length, width), dtype=int)
    index= np.random.choice(total,amount,replace=False)
    matrix[np.unravel_index(index, (length,width))]=1
    matrix= np.expand_dims(matrix, axis=2)
    return matrix

#This function places the found amound of cels randomly around the electrodes
def place_value_on_zeros(matrix, fraction):
    result = matrix.squeeze()  # (1250, 1200, 1) → (1250, 1200)
    result = result.copy()
    
    zero_mask = result == 0
    rows, cols = zero_mask.nonzero()
    
    n_twos = int(len(rows) * fraction)
    chosen_indices = np.random.choice(len(rows), size=n_twos, replace=False)
    
    result[rows[chosen_indices], cols[chosen_indices]] = 2
    return result[..., np.newaxis]  # Turns the shape back to it's original size (1250, 1200, 1)

#Reverses the 2 values to 1 values and the 1 values to zero
def replacing_2_0(matrix):
    data_new= np.where(matrix==1,0,matrix)
    data_new= np.where(data_new==2,1,data_new)
    return data_new

#combines the last two functions to define a homogeneous matrix
def random_elektrode_matrix(data2, fraction_cells):
    result= place_value_on_zeros(data2, fraction_cells)
    result2= replacing_2_0(result)
    return(result2)

#This function was used when the electrodes should not be considerd
# def maak_heterogeen(matrix, concentratie):
#     result= matrix.squeeze()
#     result = result.copy()
    
#     # Bereken blokgrootte linksboven
#     n_twos = int(matrix.size * concentratie)
#     blok = int(np.sqrt(n_twos))
    
#     # Alleen 0-posities binnen het blok linksboven
#     zero_mask = result[:blok, :blok] == 0
#     rows, cols = zero_mask.nonzero()
    
#     # Vul ze allemaal in (zijn al beperkt tot het blok)
#     result[rows, cols] = 2
    
#     return result[..., np.newaxis]

#This function places the cells in a clumb in the upperleft corner around the electrodes
def maak_heterogeen(matrix, fraction):
    matrix = matrix.copy()
    rows, cols = matrix.shape
    
    n_black = rows*cols
    target = int(n_black * fraction)
    
    #Finds all the connected black area's
    labeled, n_components = ndimage.label(matrix == 0)
    
    #Sorts the area's from mean distance till (0,0)
    priorities = []
    for i in range(1, n_components + 1):
        pixels = np.argwhere(labeled == i)
        avg_dist = np.mean(np.sqrt(pixels[:,0]**2 + pixels[:,1]**2))
        priorities.append((avg_dist, i, len(pixels)))
    priorities.sort()
    
    #Try's every erea
    for avg_dist, region_id, size in priorities:
        if size >= target:
            pixels = np.argwhere(labeled == region_id)
            pixel_dists = np.sqrt(pixels[:,0]**2 + pixels[:,1]**2)
            start = pixels[np.argmin(pixel_dists)]
            
            visited = np.zeros_like(matrix, dtype=bool)
            queue = []
            heapq.heappush(queue, (pixel_dists.min(), start[0], start[1]))
            visited[start[0], start[1]] = True
            count = 0
            
            while queue and count < target:
                dist, r, c = heapq.heappop(queue)
                matrix[r, c] = 2
                count += 1
                
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols:
                        if not visited[nr, nc] and labeled[nr, nc] == region_id:
                            visited[nr, nc] = True
                            d = np.sqrt(nr**2 + nc**2)
                            heapq.heappush(queue, (d, nr, nc))
            
            return matrix
    
    print("Past niet: geen enkel zwart gebied is groot genoeg")
    return matrix

#This makes the heterogenuous matrix by making the matrix and converting the values
def hetrogreen_matrix(data2, fraction_cells):
    hetrogeen= np.expand_dims(maak_heterogeen(data2.squeeze(), fraction_cells),axis=-1)
    hetrogeen2= replacing_2_0(hetrogeen)
    return hetrogeen2

#This function is the function that calculates the standarddeviation for ever size of the macropixel
def method_franca(matrix,length,width):
    all_std=[]
    I=matrix.squeeze()
    S = np.pad(np.cumsum(np.cumsum(I, axis=0), axis=1), ((1,0),(1,0)), mode='constant')

    sizes = np.arange(1, min(length, width) + 1)
    r_values = (sizes ** 2) / (length * width)

    for size in range(1, min(length, width) + 1):
        sommen = S[size:, size:] - S[:-size, size:] - S[size:, :-size] + S[:-size, :-size]
        mean_values = sommen / (size * size)
        
        all_std.append(np.std(mean_values))
    return all_std, r_values

#This function simulates 20 homogenuos situations and calculates the mean standarddeviation per size of the macropixel
def homogeen(data2,fraction_cells,length,width):
    Sw_matrix=[]
    r_matrix=[]
    for i in range(20):
        matrix= random_elektrode_matrix(data2,fraction_cells)
        # segmentatie_plot(matrix)
        Sw, r= method_franca(matrix.squeeze(),length,width)
        Sw_matrix.append([Sw])
        r_matrix.append([r])
        Sw_homo= np.mean(Sw_matrix,axis=0)
        r_homo= np.mean(r_matrix,axis=0 )
    return Sw_homo, r_homo

def plot_table_output(dataset):
    colors = np.array([
        [[255,156,205] ,  [228,79,109],   [48,203,255],[255,232,118]],      
        [[165,165,165],   [182,82,224],   [159,149,223], [85, 160, 118] ],     
        [[255, 183, 52],[255, 130, 108],    [242, 162, 242], [62, 255, 255]],         
        [[172, 89, 89],[85, 170, 170],[108, 218, 108],[128, 76, 167]]])

    labels = dataset


    fig, ax = plt.subplots()

    ax.imshow(colors)

    #Sets the ticks on the x and y axis on the right places
    rows, cols = labels.shape
    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))

    # setting the labels for the rows and columns
    ax.set_xticklabels([1,2,3,4])
    ax.set_yticklabels([4,3,2,1])

  #puts the tekst in the right place
    for i in range(rows):
        for j in range(cols):
            ax.text(j, i, labels[i, j],
                    ha="center", va="center",
                    color="black", fontsize=12)

    # Aslijnen uitzetten voor een strakke tabel
    ax.spines[:].set_visible(False)
    ax.set_xlabel("Well kolommen")
    ax.set_ylabel('Well rijen')
    plt.title(f"{name}")
    #turn this on if you want to save your output
    #plt.savefig(f'{output_path}\{name}_{well}_{day}.png', pad_inches=0)
    plt.show()



In [ ]:
#This code blok is for one picture
#This path loads the cell segmentation matrix
path= r"insert your path to the cell segmentation" #The r stands for a read string. This is needed because of the \ in a path
#This path is used to store the output
output_path= r"insert your path to the map where you want the output to be saved in" #The r stands for a read string. This is needed because of the \ in a path

with h5py.File(f"{path}") as f:
        data = f["exported_data"][:]    # loads the dataset as a numpy array
        data= np.where(data==2,0,data)      #In Ilastik the background gets the label 2 here we want the background set to zero

#I Used this to check if my images keeps the same dimensions and define things as the lenght and width
print(data.shape)
#I used name in my plt.savefig() function to make sure every situation gets a different and name and is saved correctly
name = 'cellen'
segmentatie_plot(data)
#Defining the length, width, total area and the fraction of found cells
length, width, total, fraction_cells= fraction(data)
print(fraction_cells)

#This path is a path loads the electrode segmentation
path2= r"insert the path to your electrode segmenation" #The r stands for a read string. This is needed because of the \ in a path

with h5py.File(f"{path2}") as f:
        data2 = f["exported_data"][:]    #loads the dataset as a numpy array
        data2= np.where(data2==2,0,data2) #In Ilastik the background gets the label 2 here we want the background set to zero

segmentatie_plot(data2)
segmentatie_plot(random_elektrode_matrix(data2, fraction_cells))

#removes the last demention so it can be used in the functions
data_new= data.squeeze()

#Returning the standarddeviation and the metric r of the sample situation so it can be plotted
Sw_sample,r_sample= method_franca(data_new,length, width)

#This plots the sample situation
plt.figure(figsize=(8, 5))
plt.plot(r_sample, Sw_sample, linewidth=3)
plt.xlabel('r = w² / (I×J)')
plt.ylabel('Sw')
plt.title('Homogeniteitscurve - França methode')
plt.grid(True)
plt.xlim(0, 1)
plt.show()

#Here the mean standarddeviation and the r metric of the homogeneous sitation are defined
Sw_homo, r_homo= homogeen(data2,fraction_cells,length,width)

#This plots the homogeneous situation
plt.figure(figsize=(8, 5))
plt.plot(r_homo.squeeze(), Sw_homo.squeeze(), linewidth=3)
plt.xlabel('r = w² / (I×J)')
plt.ylabel('Sw')
plt.title('Homogeniteitscurve - França methode')
plt.grid(True)
plt.xlim(0, 1)
plt.show()

#Defining the hetrogeneous matrix
I_het = hetrogreen_matrix(data2,fraction_cells )
segmentatie_plot(I_het)
#Checking if the found fraction of cells is the same as in the original situation
length2, width2, total2, fraction_cells2= fraction(I_het)
print(fraction_cells2)

#Here the standarddevitions and the mertric r are defined for the heterogeneous situation
Sw_hetro, r_hetro= method_franca(I_het, length,width)

#Here the hetrogeneous situation is plotted
plt.figure(figsize=(8, 5))
plt.plot(r_hetro, Sw_hetro, linewidth=3)
plt.xlabel('r = w² / (I×J)')
plt.ylabel('Sw')
plt.title('Homogeniteitscurve - França methode')
plt.grid(True)
plt.xlim(0, 1)
plt.show()

#There area under the curve is calculated for every situation
AUC_sample= np.trapezoid(Sw_sample,r_sample)
AUC_homo= np.trapezoid(Sw_homo,r_homo)
AUC_hetro= np.trapezoid(Sw_hetro,r_hetro)

#The homogeneouspercentage is calculated 
per_H= float((AUC_sample-AUC_hetro)/(AUC_homo-AUC_hetro)*100)
print((per_H))

#All curves are plotted in the same plot
plt.figure(figsize=(8, 5))
plt.plot(r_hetro, Sw_hetro, label='hetrogeen')
r_homo1=r_homo.squeeze()                
Sw_homo1= Sw_homo.squeeze()
plt.plot(r_homo1,Sw_homo1, label='homogeen')
plt.plot(r_sample, Sw_sample,label= 'sample')
plt.xlabel('r = w² / (I×J)')
plt.ylabel('Sw')
plt.title(f'Homogeniteitscurve - %H: {round(per_H,2)} - concentratie: {round(fraction_cells,3)}')
plt.grid(True)
plt.xlim(0, 1)
plt.legend()
plt.show()

In [ ]:
#This code blok calculate the percentages for a bulk
#Here the wells are listed that are included
wells=['A1','A2','A3','B1','B2','B3','C1','C2','C3','C4','C5','C6','D1','D2','D3','D4','D5','D6']
#here the days that are measured are listed
days=[1,2,4,8]

#In this dubble for loop are the homogeneouspercentages calculated for every measure day
for well in wells:
        for day in days:
            path= r"insert the path to the map where your celsegmentations are saved" #The r stands for a read string. This is needed because of the \ in a path
            path= os.path.join(path, f"{well}\cropped_{well}_{day}_elektrode_cellen_Simple_Segmentation.h5") #This how i named my files. If you choose to do otherwise rember to change this line
            output_path= r"insert the path to your map where you want the output to be saved " #The r stands for a read string. This is needed because of the \ in a path
            output_path= os.path.join(output_path,f"{well}")
            #Openen sample dataset
            with h5py.File(f"{path}") as f:
                data = f["exported_data"][:]    # laad dataset als NumPy array
                data= np.where(data==2,0,data)
            print(data.shape)
            name= 'cellen'
            segmentatie_plot(data)
            length, width, total, fraction_cells= fraction(data)
            print(fraction_cells)

            #openen electrodes
            path2= r"Insert the path to the map where the electrode segmentations are saved" #The r stands for a read string. This is needed because of the \ in a path
            path2= os.path.join(path2,f"{well}\cropped_{well}_{day}_elektrode_Simple_Segmentation.h5") #This how i named my files. If you choose to do otherwise rember to change this line

            with h5py.File(f"{path2}") as f:
                data2 = f["exported_data"][:]    # laad dataset als NumPy array
                data2= np.where(data2==2,0,data2)
            name='elektrode'
            segmentatie_plot(data2)
            name= 'homogeen'
            segmentatie_plot(random_elektrode_matrix(data2, fraction_cells))

            data_new= data.squeeze()

            Sw_sample,r_sample= method_franca(data_new,length, width)

          

            Sw_homo, r_homo= homogeen(data2,fraction_cells,length,width)

            I_het = hetrogreen_matrix(data2,fraction_cells )
            name='heterogeen'
            segmentatie_plot(I_het)
            length2, width2, total2, fraction_cells2= fraction(I_het)
            print(fraction_cells2)



            Sw_hetro, r_hetro= method_franca(I_het, length,width)


            AUC_sample= np.trapezoid(Sw_sample,r_sample)
            AUC_homo= np.trapezoid(Sw_homo,r_homo)
            AUC_hetro= np.trapezoid(Sw_hetro,r_hetro)

            per_H= float((AUC_sample-AUC_hetro)/(AUC_homo-AUC_hetro)*100)
            print((per_H))

            plt.figure(figsize=(8, 5))
            plt.plot(r_hetro, Sw_hetro, label='heterogeen')
            r_homo1=r_homo.squeeze()
            Sw_homo1= Sw_homo.squeeze()
            plt.plot(r_homo1,Sw_homo1, label='homogeen')
            plt.plot(r_sample, Sw_sample,label= 'sample')
            plt.xlabel('r')
            plt.ylabel('Sw')
            plt.title(f'Homogeniteitscurve - %H: {round(per_H,2)} - concentratie: {round(fraction_cells,3)}')
            plt.grid(True)
            plt.xlim(0, 1)
            plt.legend()
            plt.savefig(f"{output_path}\Homogeinieits_score_{well}_{day}.png", bbox_inches= 'tight')
            plt.show()

In [ ]:
#This is how i calculated the homogeneouspercentages for every electrode in a well
#Defining the wells you want to look at
wells=['A1','A2','A3','B1','B2','B3','C1','C2','C3','C4','C5','C6','D1','D2','D3','D4','D5','D6']
#the days that are measured
days=[1,2,4,8]
#The amount of rows
rijën=[1,2,3,4]
#the amount of columns
kolommen=[1,2,3,4]
#a list to save the total fraction of found cells
fracties_cellen_totaal=[]
#a list that saves the fraction that covers up the electrode
fracties_elektrode_totaal=[]
#A list that saves the found homogeneouspercentages
percentages=[]
#a list that saves the names of the electrode to check if it is plotted correctly
namen=[]


for well in wells:
    for day in days:
        fracties_cellen_totaal=[]
        fracties_elektrode_totaal=[]
        percentages=[]
        namen=[]
        for rij in rijën:
            rij_namen=[]
            rij_percentages=[]
            fracties_cellen_rij=[]
            fracties_elektrode_rij=[]
            for kolom in kolommen:
                rij_namen.append(f'r{rij}k{kolom}')
                path= r"insert the path to the map where your celsegmentations are saved" #The r stands for a read string. This is needed because of the \ in a path
                path= os.path.join(path, f"{well}\{well}_{day}_r{rij}k{kolom}_cellen_Simple_Segmentation.h5") #This how i named my files. If you choose to do otherwise rember to change this line
                output_path= r"insert the path to your map where you want the output to be saved " #The r stands for a read string. This is needed because of the \ in a path
                output_path= os.path.join(output_path,f"{well}_elektrode") #This how i named my files. If you choose to do otherwise rember to change this line
                #Openen sample
                with h5py.File(f"{path}") as f:
                    data = f["exported_data"][:]    # laad dataset als NumPy array
                    data= np.where(data==2,0,data)
                print(data.shape)
                name= 'cellen'
                segmentatie_plot(data)
                length, width, total, fraction_cells= fraction(data)
                print(fraction_cells)
                fracties_cellen_rij.append(round(fraction_cells,3))


                #openen elektrodes
                path2= r"Insert the path to the map where the electrode segmentations are saved"  #The r stands for a read string. This is needed because of the \ in a path
                path2= os.path.join(path2,f"{well}\{well}_{day}_r{rij}k{kolom}_elektrode_Simple_Segmentation.h5") #This how i named my files. If you choose to do otherwise rember to change this line

                with h5py.File(f"{path2}") as f:
                    data2 = f["exported_data"][:]    
                    data2= np.where(data2==2,0,data2)
                    
                name='elektrode'
                segmentatie_plot(data2)
                length3, width3, total3, fraction_cells3= fraction(data2)
                fracties_elektrode_rij.append(round(fraction_cells3,3))
                name= 'homogeen'
                segmentatie_plot(random_elektrode_matrix(data2, fraction_cells))

                data_new= data.squeeze()

                Sw_sample,r_sample= method_franca(data_new,length, width)


                Sw_homo, r_homo= homogeen(data2,fraction_cells,length,width)


                I_het = hetrogreen_matrix(data2,fraction_cells )
                name='heterogeen'
                segmentatie_plot(I_het)
                length2, width2, total2, fraction_cells2= fraction(I_het)
                print(fraction_cells2)


                Sw_hetro, r_hetro= method_franca(I_het, length,width)


                AUC_sample= np.trapezoid(Sw_sample,r_sample)
                AUC_homo= np.trapezoid(Sw_homo,r_homo)
                AUC_hetro= np.trapezoid(Sw_hetro,r_hetro)

                per_H= float((AUC_sample-AUC_hetro)/(AUC_homo-AUC_hetro)*100)
                print((per_H))
                rij_percentages.append(round(per_H,2))

                plt.figure(figsize=(8, 5))
                plt.plot(r_hetro, Sw_hetro, label='heterogeen')
                r_homo1=r_homo.squeeze()
                Sw_homo1= Sw_homo.squeeze()
                plt.plot(r_homo1,Sw_homo1, label='homogeen')
                plt.plot(r_sample, Sw_sample,label= 'sample')
                plt.xlabel('r')
                plt.ylabel('Sw')
                plt.title(f'Homogeniteitscurve - %H: {round(per_H,2)} - concentratie: {round(fraction_cells,3)}')
                plt.grid(True)
                plt.xlim(0, 1)
                plt.legend()
                plt.savefig(f"{output_path}\Homogeinieits_score_{well}_{day}_r{rij}k{kolom}.png", bbox_inches= 'tight')
                plt.show()
            namen.insert(0,rij_namen)
            fracties_cellen_totaal.insert(0,fracties_cellen_rij)
            fracties_elektrode_totaal.insert(0,fracties_elektrode_rij)
            percentages.insert(0,rij_percentages)
        name='fractie gevonden cellen'
        plot_table_output(np.array(fracties_cellen_totaal))
        name='fractie dat de elektrode bedekt'
        plot_table_output(np.array(fracties_elektrode_totaal))
        name= 'homogeniteitspercentage'
        plot_table_output(np.array(percentages))
        print(f'{well} dag {day} gehad.')